# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmedosrf/flyrank-ml-internship-ahmedosrf/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My provisional lane is **Lane 2 — Refresh / Content Opportunity Scoring**. The ML task is **ranking/scoring**: assign each page a priority score and rank pages so an editor can review a limited top-K queue first. A classifier can be used as the scoring engine because its probability for the observed decline label gives a comparable score, but the business output is the ranked queue rather than a yes/no automation. The task is therefore evaluated as prioritization, not as a claim that a page will certainly decline or that a refresh will certainly recover it.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data/raw/content_refresh_anonymized.csv").exists():
    ROOT = Path("/home/ubuntu/flyrank-ml-internship-starter")
DATA_PATH = ROOT / "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = df["trend_direction"].eq("down").astype(int)
print({"rows": len(df), "clients": df["client_id"].nunique(), "target_rate": round(df["is_declining_label"].mean(), 3)})


{'rows': 30000, 'clients': 32, 'target_rate': np.float64(0.542)}


## 2. Target or proxy

The starter proxy target is `is_declining_label`, equal to 1 when the current row's `trend_direction` is `down`. This is an **observed snapshot label** derived from the comparison between the latest and previous 30-day impression windows; it is not a future causal outcome and it does not prove that a page needs a refresh. The eventual scoring output would be a page-level probability or priority score for this label, while `trend_direction` and `trend_pct` must be excluded from features because they define or directly encode the target. The target is useful for the first exercise because it lets us test the mechanics of ranking, but the limitation must remain visible when interpreting the result.

In [2]:
# The target is a snapshot proxy, not a future outcome.
print({
    "target_column": "is_declining_label",
    "positive_rows": int(df["is_declining_label"].sum()),
    "positive_rate": round(float(df["is_declining_label"].mean()), 3),
    "label_source": "trend_direction == 'down'",
})
assert "trend_direction" not in {"search_volume", "impressions_90d", "avg_position", "ctr", "days_since_last_update"}


{'target_column': 'is_declining_label', 'positive_rows': 16262, 'positive_rate': 0.542, 'label_source': "trend_direction == 'down'"}


## 3. Success metric

The primary success metric is **Precision@50 on a client-grouped holdout**. It asks: among the 50 pages placed at the top of the review queue for clients not used for training, what fraction have the observed decline label? This matches the real capacity constraint better than accuracy: the editor has a finite review budget and needs the top of the queue to be useful. I will compare the model score with a transparent fixed-rule baseline on the same split. A model is worth keeping only if it improves or meaningfully complements that baseline without hiding the reason for a high score. I will report the result as decision-support performance, not as a guaranteed business lift.

In [3]:
# Precision@50 is a ranking metric: it measures the observed positive rate in the top queue.
def precision_at_k(y_true, score, k=50):
    ranked = pd.DataFrame({"y": y_true.to_numpy(), "score": score.to_numpy()}).sort_values("score", ascending=False).head(k)
    return float(ranked["y"].mean())

baseline_score = (
    (df["impressions_90d"] >= 500).astype(int)
    + (df["days_since_last_update"] >= 180).astype(int)
    + (df["avg_position"].between(1, 20)).astype(int)
)
print({"metric": "Precision@50", "illustrative_full_snapshot_rule_value": round(precision_at_k(df["is_declining_label"], baseline_score), 3)})


{'metric': 'Precision@50', 'illustrative_full_snapshot_rule_value': 0.7}


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one pseudonymized content page at the exported snapshot**, not one client, query, day, or user. Each row contains the page's trailing-90-day activity, content properties, recent movement fields, and client identifier for grouping. The code below loads the starter CSV, constructs the observed proxy label, selects a compact lane slice, and displays actual rows so the grain is inspectable.

In [4]:
lane_slice = df[[
    "content_id", "client_id", "content_type", "impressions_90d", "ctr",
    "avg_position", "content_age_days", "days_since_last_update", "is_declining_label"
]].copy()
print("One row = one pseudonymized content page at the exported snapshot")
lane_slice.head(8)


One row = one pseudonymized content page at the exported snapshot


,content_id,client_id,content_type,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,0.76,10.6,187,20,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,0.05,20.3,445,25,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,0.09,36.5,141,20,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,0.49,6.2,463,22,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,0.13,44.0,263,14,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,0.03,8.5,147,20,1
6,content_9a34b442b552,client_8722616204,keyword article,20,0.00,7.0,90,20,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,0.06,21.2,445,22,0


## 5. Why ML beats a fixed rule here

A fixed rule such as “stale and visible” is valuable as a transparent baseline and may be enough if it performs well. ML earns a place only because page priority may depend on several interacting signals: impressions, average position, CTR, freshness, content age, engagement, and demand. A single threshold can miss pages that have different combinations of exposure and decay, while a model can learn a smoother ranking and expose feature contributions for review. This is a conditional claim: I will prefer the fixed rule if it performs as well or is easier to act on. The model must be trained without the label-defining trend fields, evaluated on clients held out from training, and accompanied by reason codes or a reviewer check.

In [5]:
# Self-check: page grain, proxy target, and leakage-sensitive fields are explicit.
assert lane_slice["content_id"].nunique() == len(lane_slice)
assert lane_slice.shape[1] == 9
assert lane_slice["is_declining_label"].isin([0, 1]).all()
assert "trend_pct" not in lane_slice.columns
print("Self-check passed: page-level dataframe, explicit proxy target, Precision@50, and no trend_pct feature.")


Self-check passed: page-level dataframe, explicit proxy target, Precision@50, and no trend_pct feature.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.